### Phase 1: Datenbeschaffung & Preprocessing
Zuerst benötigen wir die Rohdaten. Der Zeitraum von mindestens 20-30 Jahren ist ideal (um mehrere Zyklen wie 2000, 2008 und 2020 abzudecken).

*   **Libraries:** `yfinance`, `pandas`, `numpy`
*   **Datenquellen:**
    *   **Risiko-Asset:** 60% S&P 500 (Ticker: `^GSPC`), 40% US-Staatsanleihen (Ticker: `VUSTX`)
    *   **Safe-Haven:** 3-Monats-Treasury Bill Rate als Proxy für Cash-Zinsen(Ticker: `^IRX`)
    *   **Features:** VIX Index (Volatilität), Renditestrukturkurve (10Y-2Y Spread), Momentum-Indikatoren.

In [ ]:
# --- Central Config ---
import sys; sys.path.insert(0, "../config")
from config_loader import cfg

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# --- Ingestion aus src/ ---
sys.path.insert(0, "..")
from src.data.ingestion import download_market_data, save_raw_data
from src.data.preprocessing import preprocess_pipeline

# 1. Daten laden
data = download_market_data(
    tickers=cfg.data.tickers,
    start_date=cfg.data.start_date,
    end_date=cfg.data.end_date,
)

# --- Rohdaten im Bronze-Layer sichern (vor jeglicher Bereinigung) ---
save_raw_data(data, cfg.data_path("raw"))
print(f"Rohdaten gespeichert unter: {cfg.data_path('raw')}")

# 2. Portfolio Erstellung (60% S&P 500, 40% Long Term Bonds)
df = preprocess_pipeline(
    data=data,
    weight_equity=cfg.portfolio.weight_equity,
    weight_bonds=cfg.portfolio.weight_bonds,
)

print(f"Erfolgreich geladen: {df.index[0].date()} bis {df.index[-1].date()}")
print(df)

# Explorative Datenanalyse (EDA)

In [ ]:
import scipy.stats as stats
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

In [ ]:
# 1. Deskriptive Statistik-Tabelle
print("\n--- DESKRIPTIVE STATISTIK ---")

# --- EDA aus src/ ---
from src.data.eda import calculate_descriptive_stats

cols_to_analyze = ['Returns_GSPC', 'Returns_VUSTX', 'Returns']

desc_stats_df = calculate_descriptive_stats(df, cols_to_analyze)

# Als Markdown persistieren via config.yaml Pfad
desc_stats_path = cfg.asset_path("eda_descriptive_stats")
desc_stats_df.to_markdown(desc_stats_path)

print(f"Deskriptive Statistik gespeichert unter: {desc_stats_path}")
display(desc_stats_df)

In [ ]:
# 2. Stationaritätstest (Augmented Dickey-Fuller)
print("\n--- STATIONARITÄTSTESTS (ADF) ---")

# --- EDA aus src/ ---
from src.data.eda import run_adf_test

adf_table_df = run_adf_test(df, cols_to_analyze)

# Als Markdown persistieren
adf_stats_path = cfg.asset_path("eda_adf_tests")
adf_table_df.to_markdown(adf_stats_path)

print(f"ADF-Test Tabelle gespeichert unter: {adf_stats_path}")
display(adf_table_df)

In [ ]:
# 3. Volatilitätscluster und Autokorrelation (Heteroskedastizität)
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# Plot 1: Rendite-Verlauf S&P 500
axes[0].plot(df.index, df['Returns_GSPC'], color='blue', alpha=0.6)
axes[0].set_title('S&P 500 Tägliche Renditen (Visualisierung von Volatilitätsclustern)')
axes[0].set_ylabel('Rendite')
axes[0].grid(True, alpha=0.3)

# Plot 2: ACF der quadrierten Renditen
squared_returns = df['Returns_GSPC']**2
plot_acf(squared_returns.dropna(), lags=40, ax=axes[1], alpha=0.05, color='red')
axes[1].set_title('Autokorrelation der quadrierten Renditen (S&P 500) - Nachweis von GARCH-Effekten')

plt.tight_layout()

# Plot persistieren
vol_cluster_path = cfg.asset_path("eda_volatility_clusters")
plt.savefig(vol_cluster_path, dpi=300, bbox_inches='tight')
print(f"Volatilitätscluster-Plot gespeichert unter: {vol_cluster_path}")

plt.show()

In [ ]:
# 4. Historische Drawdowns (SORR Kontext)
# Kumulierte Rendite und Peak berechnen
cum_returns = np.exp(df['Returns'].cumsum())
running_max = cum_returns.cummax()
drawdowns = (cum_returns / running_max) - 1.0

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(drawdowns.index, drawdowns, color='darkred')
ax.fill_between(drawdowns.index, drawdowns, 0, color='red', alpha=0.3)
ax.set_title('Historische Drawdowns des 60/40 Portfolios (Benchmark)')
ax.set_ylabel('Drawdown')
ax.grid(True, alpha=0.3)

# Plot persistieren
drawdown_plot_path = cfg.asset_path("eda_historical_drawdowns")
plt.savefig(drawdown_plot_path, dpi=300, bbox_inches='tight')
print(f"Drawdown-Plot gespeichert unter: {drawdown_plot_path}")

plt.show()

# Die Top 3 Krisen-Tiefpunkte identifizieren
print("\nTop 3 Tiefpunkte des 60/40 Drawdowns:")
print(drawdowns.sort_values().head(3) * 100) # In Prozent anzeigen

In [ ]:
from pathlib import Path

output_path = Path(cfg.data_path("preprocessed"))

# Verzeichnis anlegen, wenn nicht vorhanden
output_path.parent.mkdir(exist_ok=True)

# Speichern als Parquet
df.to_parquet(output_path)
print(f"Dataframe erfolgreich unter {output_path} gespeichert.")